# 🌐 Notebook: Apache Kafka — Ecosystem, Scaling & Where It Fits

The first two notebooks covered Kafka on its own terms — architecture, producing, consuming, delivery guarantees, retention. This last notebook zooms out: how Kafka compares to the other messaging systems you'll hear about in the same breath, what the wider Kafka ecosystem (Connect, Streams) actually does, how a cluster scales and survives broker failures in production, and what you'd actually watch on a dashboard once Kafka is running for real. These are exactly the "so how does this fit into the bigger picture" questions that tend to close out a technical conversation about Kafka.

> This notebook assumes the broker and Kafka UI from [Notebook 1](./02_1_intro_kafka.ipynb) are still running. If not, re-run its Docker setup cells first (section 4) before continuing here.

## 📚 Sources

- [Kafka: Replication](https://kafka.apache.org/documentation/#replication)
- [Kafka Connect Documentation](https://kafka.apache.org/documentation/#connect)
- [Kafka Streams Documentation](https://kafka.apache.org/documentation/streams/)
- [Kafka: Monitoring](https://kafka.apache.org/documentation/#monitoring)
- [Amazon SQS Documentation](https://docs.aws.amazon.com/sqs/) (for comparison)
- [RabbitMQ Documentation](https://www.rabbitmq.com/docs) (for comparison)


In [1]:
from confluent_kafka import Producer, Consumer
from confluent_kafka.admin import AdminClient, NewTopic
import json, time

BOOTSTRAP = "localhost:9092"
admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
producer = Producer({"bootstrap.servers": BOOTSTRAP})
print("connected brokers:", admin.list_topics(timeout=5).brokers)

connected brokers: {1: BrokerMetadata(1, localhost:9092)}


## 1. Kafka vs. traditional message queues

"Why not just use RabbitMQ / SQS?" is one of the first questions any Kafka adoption has to answer, and the honest answer is: it depends what you're building. All three move messages from a producer to a consumer, but they were designed around different assumptions.

**RabbitMQ** and **AWS SQS** are built around the classic **queue** model: a message is delivered to *one* consumer, and once acknowledged, it's gone — deleted from the queue. This is a great fit for **task distribution** (a pool of workers each grabbing the next job off a shared queue) and for systems that need sophisticated per-message routing (RabbitMQ's exchanges support complex topic/header-based routing rules that Kafka doesn't attempt to replicate). Throughput is typically far lower than Kafka's, and neither is designed to let you replay history — once a message is consumed and acknowledged, it's not coming back.

**Kafka** is built around the **log** model: a message is *retained* (for a configured period, or forever with compaction — see Notebook 2) and can be read by any number of independent consumer groups, each at its own pace, including groups that don't exist yet at the time the message was written. This is what makes Kafka the right choice for **event streaming** at scale: many independent systems reacting to the same stream of facts, systems that need to reprocess history, or throughput in the range of millions of messages per second.

| | Kafka | RabbitMQ | AWS SQS |
|---|---|---|---|
| Model | Distributed commit log | Message queue / broker | Managed message queue |
| Message lifecycle | Retained per policy, replayable | Deleted on ack | Deleted on ack (visibility timeout) |
| Multiple independent consumers of the same message | Yes, natively (consumer groups) | Needs fan-out exchanges | Needs SNS fan-out in front of it |
| Typical throughput | Very high | Moderate | Moderate, auto-scaled |
| Ordering | Per-partition | Per-queue (with caveats) | FIFO queues only, and capped throughput |
| Routing logic | Simple (topic + partition key) | Rich (exchanges, bindings) | Simple |
| Operating it yourself | You run/scale the cluster (or use a managed offering) | You run/scale the broker | Fully managed by AWS |

A useful rule of thumb: if the question is "how do I distribute *jobs* to a pool of workers, each job handled exactly once", reach for a queue (RabbitMQ/SQS). If the question is "how do I let many different, decoupled parts of my system react to the same stream of events, possibly re-reading history", reach for Kafka.


## 2. The wider ecosystem: Connect and Streams

Kafka itself only does one thing: store and move records reliably. Two official sub-projects build on top of that core to solve the problems that come up immediately once you actually use it.

### Kafka Connect — getting data in and out without writing a client

Writing a custom producer or consumer every time you need to move data between Kafka and a database, a cloud storage bucket, or another system is repetitive, error-prone work that has nothing to do with your actual business logic. **Kafka Connect** is a framework (that ships with Kafka) for running reusable, configuration-only integrations called **connectors**:

- A **source connector** reads from an external system and produces to a Kafka topic — e.g. a Postgres source connector using **change data capture (CDC)** to stream every row insert/update/delete into a topic in near real time, without touching the application that owns that database.
- A **sink connector** consumes from a Kafka topic and writes to an external system — e.g. streaming a topic's contents into a data warehouse or an Elasticsearch index for search.

You configure a connector with a JSON blob (which class to use, which topic, which connection details) rather than writing a Python producer/consumer loop yourself — Connect handles scaling across multiple workers, offset tracking, and retries for you.

### Kafka Streams — processing data without leaving Kafka

Once data is in Kafka, **Kafka Streams** is a Java library (part of Kafka itself, not a separate cluster to run) for building applications that transform, filter, join, or aggregate one or more topics into other topics — continuously, as new records arrive, rather than in scheduled batches. It's built on the same **stream-table duality** idea from Notebook 2's log-compaction discussion: a stream of events can always be turned into a table of "current state" by keeping only the latest value per key, and a table can be turned back into a stream of the changes made to it. **Windowing** (grouping events into e.g. "5-minute buckets" for aggregation) is a core Streams concept for anything time-based, like "count of orders per 5-minute window".

Since Kafka Streams is a JVM library rather than something with its own Python client, we won't run code for it here — but recognizing what problem it solves (continuous, stateful stream processing that reads from and writes back to Kafka, without a separate processing cluster like Spark or Flink) is exactly the kind of ecosystem knowledge that distinguishes "I've used the Kafka client" from "I understand how Kafka fits into a data platform". Frameworks like **Apache Flink** and **Spark Structured Streaming** solve a similar problem from *outside* Kafka, typically when you already have one of those clusters for other reasons or need capabilities (e.g. complex event-time handling across many external sources) beyond what a single Kafka Streams application targets.


## 3. Replication and fault tolerance

Every partition we've created so far had `replication_factor=1` — appropriate for a single-broker development setup, but a partition with only one copy is a single point of failure: lose that one broker, and the partition's data is gone. Production Kafka clusters use a replication factor of (typically) 3, meaning each partition's data is copied across 3 different brokers.

- **Leader and followers.** For each partition, one replica is elected the **leader** — all reads and writes for that partition go through it. The other replicas are **followers**, which continuously pull new records from the leader to stay in sync.
- **In-sync replicas (ISR).** A follower is considered "in-sync" as long as it hasn't fallen too far behind the leader. The set of currently in-sync replicas is the **ISR** — this is exactly what `acks=all` (Notebook 2, section 1) waits on: every replica in the ISR, not necessarily every replica that technically exists.
- **`min.insync.replicas`.** A broker-side safety net: the minimum ISR size required for a write to even be *accepted*, regardless of what the producer's `acks` setting asks for. With `replication.factor=3` and `min.insync.replicas=2`, the cluster tolerates losing one broker without any interruption to writes, and refuses writes outright (rather than silently under-replicating) if it can't maintain that minimum.
- **Failover.** If the leader for a partition crashes, one of its in-sync followers is automatically elected the new leader (this election is itself coordinated through the KRaft controller quorum from Notebook 1) — clients transparently reconnect to whichever broker is now the leader, typically within seconds.

Concretely: `replication.factor=3` with `min.insync.replicas=2` and `acks=all` on the producer is the standard "don't lose data" combination you'll see recommended almost everywhere, because it tolerates a single broker failure with zero data loss and no manual intervention. We can't fully demonstrate multi-broker failover with our single-container setup, but you can see the resulting configuration directly on any topic:


In [2]:
# Even on our single-broker cluster, topic configs like min.insync.replicas
# are visible and can be set - they just can't have practical effect above
# replication.factor=1 here.
demo_topic = "payments"
admin.create_topics([NewTopic(
    demo_topic, num_partitions=3, replication_factor=1,
    config={"min.insync.replicas": "1"},  # would be 2 alongside replication.factor=3 in production
)])
time.sleep(1)

metadata = admin.list_topics(timeout=5).topics[demo_topic]
for partition_id, partition in metadata.partitions.items():
    print(f"partition {partition_id}: leader=broker-{partition.leader}, replicas={partition.replicas}, isr={partition.isrs}")

partition 0: leader=broker-1, replicas=[1], isr=[1]
partition 1: leader=broker-1, replicas=[1], isr=[1]
partition 2: leader=broker-1, replicas=[1], isr=[1]


## 4. Monitoring a running cluster

Once Kafka is running for real, the handful of metrics worth watching are the same ones that came up implicitly throughout this chapter:

- **Consumer lag** (introduced in Notebook 2) — per partition, per consumer group. The single most common Kafka alert in production: rising lag means consumers can't keep up.
- **Throughput** — messages/bytes per second, in and out, per topic and per broker. Sudden drops often mean a downstream problem; sudden spikes can mean a misbehaving producer.
- **Under-replicated partitions** — partitions where the ISR has shrunk below the full replica count, i.e. one or more followers have fallen behind or gone offline. A leading indicator of broker trouble before it becomes data loss.
- **Request latency** (produce/fetch) at the broker level — rising latency, especially on `acks=all` writes, often points to disk I/O pressure or an undersized cluster.
- **Disk usage** per broker — since Kafka's retention policies (Notebook 2) are what stands between "steady state" and "disk full", this is a hard operational limit, not just a nice-to-have to watch.

Kafka UI, the same dashboard we've had open throughout this chapter, already surfaces consumer lag and per-topic throughput without any extra setup — in a real production deployment you'd typically export these same metrics (Kafka exposes them all via JMX) into a proper monitoring stack (Prometheus + Grafana is the most common combination) for alerting and historical dashboards, since a browser tab isn't something you page someone from.


## 5. Exercises

### Exercise 1: Choosing the right tool

**Task:** For each scenario below, decide whether Kafka, RabbitMQ/SQS, or either would be a reasonable fit, and write one sentence justifying your choice:

1. Distributing image-resizing jobs to a pool of 20 worker processes, each job handled exactly once.
2. Feeding a real-time fraud-detection system, a real-time analytics dashboard, and a data-warehouse loader from the same stream of payment events — all three need every event, independently.
3. Sending a single "password reset" email per user request.


In [3]:
# Your solution here:


<details>
<summary><b>Show Solution</b></summary>

1. **RabbitMQ/SQS.** This is a classic task queue: each job should be picked up and completed by exactly one worker, and once done, there's no need to keep it around or let anyone else re-read it.
2. **Kafka.** Three independent systems all need to react to the *same* stream of events, at their own pace, and a design like this often grows a fourth or fifth consumer later without needing to touch the producer — exactly the fan-out, replay-friendly case Kafka is built for.
3. **RabbitMQ/SQS** (or either, at this scale). A single point-to-point job with no need for replay, multiple independent consumers, or high throughput — a queue is simpler to operate for this than standing up a Kafka topic.

</details>


### Exercise 2: Reasoning about `min.insync.replicas`

**Task:** A topic has `replication.factor=3` and `min.insync.replicas=2`. Two of its three brokers are currently down. Can producers using `acks=all` still write to this topic? Explain why, in terms of the ISR.


In [4]:
# Your solution here:


<details>
<summary><b>Show Solution</b></summary>

No. With two of three brokers down, the ISR for that partition can have at most 1 member (whichever replica is still up) — below the configured `min.insync.replicas=2`. The broker will reject the write outright with a `NotEnoughReplicas` error rather than accept it and risk data loss, regardless of what the producer's `acks` setting requests. This is exactly the safety net `min.insync.replicas` is for: it refuses to *pretend* a write is durable when it can't actually guarantee that.

</details>


### Exercise 3: Connect vs. Streams vs. your own consumer

**Task:** You need to (a) continuously mirror a Postgres `orders` table into a Kafka topic as it changes, and (b) compute a continuously-updated "total revenue per hour" aggregate from that topic. For each of (a) and (b), name the Kafka ecosystem component that's the best fit, and explain why writing a custom Python consumer would be the wrong tool for at least one of them.


In [5]:
# Your solution here:


<details>
<summary><b>Show Solution</b></summary>

(a) **Kafka Connect**, specifically a Postgres source connector using change-data-capture — this is exactly the "get data in from an external system, continuously, without writing a custom client" problem Connect exists to solve, including handling offset tracking and scaling for you.

(b) **Kafka Streams** (or an external stream processor like Flink) — this needs stateful, windowed aggregation ("per hour") over a continuous stream, which is precisely what Streams' windowing support is built for. A custom Python consumer *could* technically compute this, but you'd be reimplementing windowing, state storage, and fault-tolerant recovery of that state by hand — exactly the machinery Streams already provides.

</details>


## Cleanup

This is the last notebook in the chapter — tear down both containers:


In [6]:
!docker rm -f kafka-broker kafka-ui
!docker network rm kafka-net

kafka-broker


%6|1789394064.820|FAIL|rdkafka#producer-1| [thrd:localhost:9092/1]: localhost:9092/1: Disconnected: connection closed by peer: receive 0 after POLLIN (after 1167ms in state UP)
%6|1789394064.820|FAIL|rdkafka#producer-2| [thrd:localhost:9092/1]: localhost:9092/1: Disconnected: connection closed by peer: receive 0 after POLLIN (after 217ms in state UP)
%6|1789394064.820|FAIL|rdkafka#producer-2| [thrd:localhost:9092/1]: localhost:9092/1: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%6|1789394064.820|FAIL|rdkafka#producer-1| [thrd:localhost:9092/1]: localhost:9092/1: Disconnected: connection reset by peer (after 0ms in state APIVERSION_QUERY)
%3|1789394065.002|FAIL|rdkafka#producer-1| [thrd:localhost:9092/1]: localhost:9092/1: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 0ms in state CONNECT)


kafka-ui


%3|1789394065.058|FAIL|rdkafka#producer-2| [thrd:localhost:9092/1]: localhost:9092/1: Connect to ipv6#[::1]:9092 failed: Connection refused (after 0ms in state CONNECT)


kafka-net
